# Lab 1 — Understand the curve number

**Twenty minutes.** This lab establishes the theoretical and numerical
framework used by both spatial-data pathways.

The goal is not to memorize an equation. The goal is to see what changes
when a convention is changed, what nonlinear averaging does, and which
number you would defend in a report.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cnkit import (
    S_from_CN,
    cn05_from_cn20,
    composite_runoff,
    runoff,
)


## 1. One storm, three documented curve numbers

Difficult Run has a 2019 table composite near 75.5 and a fitted
asymptotic value of 69.7. The lambda conversion below is a third number:
it describes the same response convention under lambda 0.05 rather than
0.20.


In [ ]:
P = 3.0
table_cn20 = 75.5
fitted_cn20 = 69.7
table_cn05 = float(cn05_from_cn20(table_cn20))

comparison = pd.DataFrame(
    [
        ["table", 0.20, table_cn20, float(runoff(P, table_cn20, lam=0.20))],
        ["fitted to gage", 0.20, fitted_cn20, float(runoff(P, fitted_cn20, lam=0.20))],
        ["same table response, converted", 0.05, table_cn05, float(runoff(P, table_cn05, lam=0.05))],
    ],
    columns=["basis", "lambda", "curve_number", "runoff_in"],
)
comparison.round(4)


**Interpretation prompt:** the converted curve number is lower, but it
reproduces nearly the same response because lambda and CN are a paired
calibration. Write one sentence explaining why changing lambda without
converting CN mixes two calibrations.


## 2. The weighting trap

Sixty percent connected impervious cover at CN 98 sits beside forty
percent woods at CN 55. Compute runoff from a one-inch storm three ways.


In [ ]:
q_distributed, q_weighted_cn, q_weighted_s = composite_runoff(
    1.0, [98, 55], [0.60, 0.40]
)
weighting = pd.Series(
    {
        "runoff by subarea, then area-weight": q_distributed,
        "area-weight CN, then compute runoff": q_weighted_cn,
        "area-weight S, then compute runoff": q_weighted_s,
    },
    name="runoff_inches",
)
weighting.round(4)


In [ ]:
cn_values = [98, 55]
s_values = [float(S_from_CN(cn)) for cn in cn_values]
print("CN 98: S = %.3f in, Ia = %.3f in" % (s_values[0], 0.2 * s_values[0]))
print("CN 55: S = %.3f in, Ia = %.3f in" % (s_values[1], 0.2 * s_values[1]))
print("distributed / weighted-CN ratio: %.1f" % (q_distributed / q_weighted_cn))


The wooded subarea produces zero runoff because its initial abstraction
exceeds the storm. Averaging CN first quietly treats the whole basin as
partly absorbing. The arithmetic is correct in every row; the modelling
decisions are different.


## 3. Why the disagreement shrinks in large storms


In [ ]:
storms = np.linspace(0.25, 6.0, 48)
values = np.array([composite_runoff(p, [98, 55], [0.60, 0.40]) for p in storms])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(storms, values[:, 0], lw=2.5, label="distributed runoff")
ax.plot(storms, values[:, 1], lw=2, label="weighted CN")
ax.plot(storms, values[:, 2], lw=2, label="weighted S")
ax.set(xlabel="storm depth, inches", ylabel="runoff depth, inches")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## Report-out

Bring back:

1. The three runoff depths from the weighting example.
2. Which convention you would report for a heterogeneous watershed.
3. One sentence on the evidence behind that choice.

**Source anchors:** NEH-630 Chapter 10 equations 10-1 and 10-11;
TR-55 Worksheet 2; Woodward et al. (2003), DOI
`10.1061/40685(2003)308`.
